In [1]:
import torch
from torch.profiler import profile, ProfilerActivity, record_function

# Model Testing

In [ ]:
# See if model can be initialized
from kdv import *

INIT_PARAMS = dict(
    num_solitons             = 1,
    n_hidden_layers          = 4, 
    n_neurons_per_layer      = 32, 
    activation               = nn.Tanh,
    seed                     = 42, 
    verbose                  = True,
)
model = KDV(INIT_PARAMS)

# Loss Function Testing

First step is testing if these functions both run and output the expected results

In [ ]:
from kdv_loss import *
from kdv_trainer import *

In [ ]:
weights = {
    'w_ic': 10.7,
    'w_bc': 2.0
    #'w_pde': 20.0 #missing on purpose to test update works correctly
}

domain = setup_training_domain(1000, 100, 100, model.soliton_params)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Test loss dict
losses = init_loss_list()
print(f'The loss dict: {losses}')

#Test loss weights
loss_weights = init_loss_weights(device, init_weights=None)
print(f'The loss wieghts as default: {loss_weights}')
loss_weights = init_loss_weights(device, weights)
print(f'The loss wieghts with dict: {loss_weights}')
print(f'Loss weights device: {loss_weights.device} \n')

#loss_comps = torch.tensor([0.3, 4.3, 1.0], device=loss_weights.device) #dummy var, check loss_components function with proper neural network
loss_comps = loss_components(model.neural_net, domain=domain)
total_loss = compute_total_loss(loss_weights, loss_comps)
#make sure that the dot operator works as intended
print(f'Total loss: {total_loss}')

The loss dict: {'total': [], 'initial': [], 'boundary': [], 'pde': []}
The loss wieghts as default: tensor([1., 1., 1.], device='cuda:0')
The loss wieghts with dict: tensor([10.7000,  2.0000,  1.0000], device='cuda:0')
Loss weights device: cuda:0 

Total loss: 12.8100004196167


# Train Function Testing

In [ ]:
#See if model can be initialized

# Loss Function Profiling

In [ ]:
def test_loss_run():
    #add the sequence of loss function calls you want to run here
    return

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], 
    record_shapes=True, 
    profile_memory=True
    with_modules=True
) as p:
    test_loss_run()

print('Non-compile: \n')
print(p.key_averages().table(sort_by='self_cuda_time_total', row_limit=-1))

torch.compile(test_loss_run)
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], 
    record_shapes=True, 
    profile_memory=True
    with_modules=True
) as p:
    test_loss_run()

print('Compile: \n')
print(p.key_averages().table(sort_by='self_cuda_time_total', row_limit=-1))

